# Polity5 Post-1990 Democratizers Panel (1990–2018, 38 Countries)

This notebook demonstrates the **polity5_democratizers** dataset: 1,046 country-year observations for 38 post-1990 democratizer countries drawn from the [Polity5 Project](https://www.systemicpeace.org/polityproject.html) (p5v2018).

**What this artifact does:**
- Identifies post-1990 democratizer countries using Polity5 polity2 scores (countries with polity2 ≤ 0 in 1985–1994 AND polity2 ≥ 6 in 1990–2010)
- Builds a structured panel dataset: each observation is one country-year
  - **Input**: JSON of 12 Polity5 component scores (democ, autoc, xrreg, xrcomp, xropen, xconst, parreg, parcomp, exrec, exconst, polcomp, durable)
  - **Output**: polity2 score as a string integer (−10 to +10)
- Used as the outcome variable in the SDET triple-interaction difference-in-differences experiment on education, inequality, and democratic erosion

**Data**: `mini_demo_data.json` — a curated subset of 39 examples spanning 13 countries, 3 time periods (1990–94, 2000–04, 2010–14), and polity2 scores from −7 to +10.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# pandas, matplotlib, seaborn, numpy — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import json
import math
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-395f4e-education-inequality-and-democratic-eros/main/round-4/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print("Loaded dataset:", data["datasets"][0]["dataset"])
print("Examples:", len(data["datasets"][0]["examples"]))

Loaded dataset: polity5_democratizers
Examples: 39


## Config

Parameters controlling what to display and analyze in this demo. The full dataset has 1,046 examples; set `N_EXAMPLES = None` to use all loaded examples.

In [5]:
# Limit to first N_EXAMPLES for a quick run; set to None to use all
N_EXAMPLES = None  # original: None (use all 1,046 examples in full dataset)

# Number of countries to show in per-country bar chart
N_COUNTRIES_PLOT = 10  # original: show all countries

## Helper Functions

These are copied directly from `data.py`. `_period_label` maps a year to the 5-year bin used in the SDET DD design. `_fix_nan` makes the output JSON-safe by replacing float NaN with None.

In [6]:
# 5-year period bins used in the SDET DD design
PERIODS = [
    ("1990-94", 1990, 1994),
    ("1995-99", 1995, 1999),
    ("2000-04", 2000, 2004),
    ("2005-09", 2005, 2009),
    ("2010-14", 2010, 2014),
    ("2015-19", 2015, 2019),
    ("2020-22", 2020, 2022),
]


def _period_label(year: int) -> str | None:
    for label, lo, hi in PERIODS:
        if lo <= year <= hi:
            return label
    return None


def _fix_nan(obj: object) -> object:
    """Recursively replace float NaN with None (JSON-safe)."""
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, dict):
        return {k: _fix_nan(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_fix_nan(v) for v in obj]
    return obj

## Step 1: Build Examples from Loaded Data

`build_polity5_examples` converts each country-year row into a structured example. The 12 Polity5 component scores become the JSON input string; polity2 becomes the string output. This mirrors the original `data.py` logic, but uses the pre-loaded `data` variable instead of the raw Excel file.

In [7]:
def build_polity5_examples(p5: pd.DataFrame) -> list[dict]:
    """One example per country-year row; input = component scores, output = polity2."""
    # Component columns that actually form polity2
    feature_cols = ["democ", "autoc", "xrreg", "xrcomp", "xropen", "xconst",
                    "parreg", "parcomp", "exrec", "exconst", "polcomp", "durable"]

    # Keep rows where polity2 is a valid number
    valid = p5[p5["polity2"].between(-10, 10)].copy()
    print(f"Polity5 valid rows (polity2 in [-10,10]): {len(valid)}")

    examples = []
    for _, row in valid.iterrows():
        features = {}
        for col in feature_cols:
            v = row.get(col)
            if v is not None and not (isinstance(v, float) and math.isnan(v)):
                features[col] = int(v) if isinstance(v, float) and v == int(v) else v

        period = _period_label(int(row["year"]))
        inp = json.dumps(features)
        out = str(int(row["polity2"])) if row["polity2"] == int(row["polity2"]) else str(round(float(row["polity2"]), 2))

        examples.append({
            "input": inp,
            "output": out,
            "metadata_country": str(row["country"]),
            "metadata_scode": str(row["scode"]),
            "metadata_year": int(row["year"]),
            "metadata_period": period,
            "metadata_ccode": int(row["ccode"]) if pd.notna(row.get("ccode")) else None,
        })

    print(f"Built {len(examples)} Polity5 examples")
    return examples

## Step 2: Convert Loaded Examples to a DataFrame

The pre-built examples from `mini_demo_data.json` are already in the output schema. We parse the input JSON strings and expand them into a flat DataFrame for analysis.

In [8]:
examples = data["datasets"][0]["examples"]

# Apply N_EXAMPLES limit if set
if N_EXAMPLES is not None:
    examples = examples[:N_EXAMPLES]

print(f"Working with {len(examples)} examples")

# Convert to flat DataFrame: parse input JSON strings, add metadata columns
rows = []
for ex in examples:
    features = json.loads(ex["input"])
    row = {**features}
    row["polity2"] = int(ex["output"])
    row["country"] = ex["metadata_country"]
    row["scode"] = ex["metadata_scode"]
    row["year"] = ex["metadata_year"]
    row["period"] = ex["metadata_period"]
    row["ccode"] = ex["metadata_ccode"]
    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
print(f"Countries: {df['country'].nunique()}")
print(f"Years: {df['year'].min()}–{df['year'].max()}")
df.head()

Working with 39 examples
DataFrame shape: (39, 18)
Countries: 13
Years: 1990–2010


,democ,autoc,xrreg,xrcomp,xropen,xconst,parreg,parcomp,exrec,exconst,polcomp,durable,polity2,country,scode,year,period,ccode
0,3,2,2,2,4,3,3,3,7,3,6,0,1,Albania,ALB,1990,1990-94,339
1,6,1,2,2,4,5,3,4,7,5,8,3,5,Albania,ALB,2000,2000-04,339
2,9,0,3,3,4,7,2,4,8,7,9,13,9,Albania,ALB,2010,2010-14,339
3,7,0,2,2,4,7,2,3,7,7,7,0,7,Bulgaria,BUL,1990,1990-94,355
4,8,0,3,3,4,7,2,3,8,7,7,8,8,Bulgaria,BUL,2000,2000-04,355


## Step 3: Build Output Object

This mirrors the `main()` function in `data.py` — assembles the final output schema and applies NaN cleanup.

In [9]:
output_obj = {
    "datasets": [
        {
            "dataset": "polity5_democratizers",
            "examples": examples,
        },
    ]
}

output_obj = _fix_nan(output_obj)

total = len(examples)
print(f"Total examples: {total}")
print(f"Schema keys: {list(examples[0].keys())}")
print(f"\nSample input: {examples[0]['input']}")
print(f"Sample output: {examples[0]['output']}")
print(f"Sample metadata: country={examples[0]['metadata_country']}, year={examples[0]['metadata_year']}, period={examples[0]['metadata_period']}")

Total examples: 39
Schema keys: ['input', 'output', 'metadata_country', 'metadata_scode', 'metadata_year', 'metadata_period', 'metadata_ccode']

Sample input: {"democ": 3, "autoc": 2, "xrreg": 2, "xrcomp": 2, "xropen": 4, "xconst": 3, "parreg": 3, "parcomp": 3, "exrec": 7, "exconst": 3, "polcomp": 6, "durable": 0}
Sample output: 1
Sample metadata: country=Albania, year=1990, period=1990-94


## Results: Dataset Overview

Key statistics and visualizations of the polity5_democratizers dataset.

In [10]:
feature_cols = ["democ", "autoc", "xrreg", "xrcomp", "xropen", "xconst",
                "parreg", "parcomp", "exrec", "exconst", "polcomp", "durable"]

# --- Summary table ---
print("=== Dataset Summary ===")
print(f"  Examples:  {len(df)}")
print(f"  Countries: {df['country'].nunique()}")
print(f"  Years:     {df['year'].min()}–{df['year'].max()}")
print(f"  Periods:   {sorted(df['period'].dropna().unique())}")
print(f"  polity2 range: {df['polity2'].min()} to {df['polity2'].max()}")
print(f"  polity2 mean \u00b1 std: {df['polity2'].mean():.2f} \u00b1 {df['polity2'].std():.2f}")
print(f"  Within-country SD (avg): {df.groupby('country')['polity2'].std().mean():.2f}")

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Polity5 Post-1990 Democratizers Panel", fontsize=14, fontweight="bold")

# 1. polity2 score distribution
ax = axes[0]
ax.hist(df["polity2"], bins=range(-10, 12), color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(0, color="red", linestyle="--", linewidth=1, label="Authoritarian/Democratic boundary")
ax.set_xlabel("polity2 score")
ax.set_ylabel("Count")
ax.set_title("Distribution of polity2 Scores")
ax.legend(fontsize=8)

# 2. Observations per period
ax = axes[1]
period_order = ["1990-94", "1995-99", "2000-04", "2005-09", "2010-14", "2015-19"]
period_counts = df["period"].value_counts().reindex(period_order, fill_value=0)
ax.bar(period_counts.index, period_counts.values, color="teal", edgecolor="white", alpha=0.85)
ax.set_xlabel("5-year period")
ax.set_ylabel("Observations")
ax.set_title("Observations per Period")
ax.tick_params(axis="x", rotation=45)

# 3. Mean polity2 by country (top N)
ax = axes[2]
country_mean = df.groupby("country")["polity2"].mean().sort_values(ascending=False)
top_n = country_mean.head(N_COUNTRIES_PLOT)
colors = ["steelblue" if v >= 0 else "tomato" for v in top_n.values]
ax.barh(top_n.index[::-1], top_n.values[::-1], color=colors[::-1], edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Mean polity2")
ax.set_title(f"Mean polity2 by Country (top {N_COUNTRIES_PLOT})")

plt.tight_layout()
plt.savefig("polity5_overview.png", dpi=80, bbox_inches="tight")
plt.show()
print("Plot saved to polity5_overview.png")

# --- Feature correlation with polity2 ---
print("\n=== Feature Correlations with polity2 ===")
corrs = df[feature_cols + ["polity2"]].corr()["polity2"].drop("polity2").sort_values(ascending=False)
for feat, corr in corrs.items():
    bar = "#" * int(abs(corr) * 20)
    sign = "+" if corr >= 0 else "-"
    print(f"  {feat:<10} {sign}{abs(corr):.3f}  {bar}")

=== Dataset Summary ===
  Examples:  39
  Countries: 13
  Years:     1990–2010
  Periods:   ['1990-94', '2000-04', '2010-14']
  polity2 range: -7 to 10
  polity2 mean ± std: 5.23 ± 4.57
  Within-country SD (avg): 4.48


Plot saved to polity5_overview.png

=== Feature Correlations with polity2 ===
  democ      +0.977  ###################
  exrec      +0.889  #################
  xconst     +0.884  #################
  exconst    +0.884  #################
  polcomp    +0.878  #################
  xrcomp     +0.851  #################
  parcomp    +0.845  ################
  xrreg      +0.693  #############
  xropen     +0.224  ####
  durable    +0.037  
  parreg     -0.133  ##
  autoc      -0.932  ##################


/tmp/claude-1000/claude-1000/ipykernel_2717872/353214925.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
